# Kernel cuantico para QSVM

Construye el kernel de fidelidad ``K(x_i, x_j) = P(00...0)`` del circuito
``U(x_j)^dagger U(x_i)``, con el feature map elegido en `FEATURE_MAP`
(ZZ, Pauli Z+YY o Ry-CX-Rx) definido en pytket y ejecutado via guppy.

Flujo: cargar datos -> inspeccionar circuitos (sin shots) -> enviar y guardar
K_train (para `SVC.fit`) -> enviar y guardar K_test (para `SVC.predict`).
Toda la logica vive en `funciones_nexus.py`; aqui solo quedan los parametros
y las llamadas.

La construccion esta apagada por defecto (`RUN_MATRIX = False`): revisa
circuitos y costo antes de encenderla.

## 1. Configuracion y datos

Carga el dataset escalado, consulta la hoja `variables` y conserva solo las
columnas registradas en `_vars_` antes de separar los datos que alimentan los
circuitos.

In [ ]:
import pandas as pd
from IPython.display import display
from pytket.circuit.display import render_circuit_jupyter

from funciones_nexus import (
    cargar_datos_kernel_muestreo,
    obtener_feature_map,
    seleccionar_par_kernel,
    iniciar_matriz_kernel,
    iniciar_matriz_kernel_test,
    consultar_matriz_nexus,
    guardar_kernel_qsvm,
    FEATURE_MAPS,
    NIVELES_MUESTREO,
    MATRIX_BACKEND_OPTIONS,
)

# Feature map del kernel: elige una de las 3 opciones de FEATURE_MAPS.
#   "zz"       -> ZZFeatureMap (fases Z + interacciones ZZ)
#   "zyy"      -> Pauli Z+YY explicito (entrelazamiento lineal)
#   "ry_cx_rx" -> Ry -> cadena CX -> Rx
# El mismo FEATURE_MAP gobierna K_train y K_test (deben coincidir).
FEATURE_MAP = "zyy"

PROJECT_NAME = "prueba_migracion"                        # Proyecto de Nexus (se crea si no existe)
KERNEL_DATA_PATH = "data/processed/dataset_v1.xlsx"      # Excel v1 con las hojas "muestreos" y "variables"
VARIABLES_SHEET = "variables"
VARIABLES_COLUMN = "_vars_"

# Nivel de muestreo jerarquico, independiente por particion (ver hoja
# "muestreos" en dataset_v1.xlsx). Notacion del usuario -> NIVELES_MUESTREO:
#   "3"     -> solo la submuestra mas chica  (train=16, test=8)
#   "3+2"   -> union de las dos mas chicas   (train=32, test=16)
#   "3+2+1" -> la muestra completa           (train=64, test=32)
TRAIN_MUESTREO = "3+2+1"
TEST_MUESTREO = "3+2+1"

kernel_df, kernel_feature_columns, kernel_train_df, kernel_test_df = cargar_datos_kernel_muestreo(
    KERNEL_DATA_PATH,
    nivel_train=NIVELES_MUESTREO[TRAIN_MUESTREO],
    nivel_test=NIVELES_MUESTREO[TEST_MUESTREO],
)

# La hoja "variables" es la fuente de verdad para las features del kernel.
variables_df = pd.read_excel(KERNEL_DATA_PATH, sheet_name=VARIABLES_SHEET)
if VARIABLES_COLUMN not in variables_df.columns:
    raise KeyError(
        f'La hoja "{VARIABLES_SHEET}" debe contener la columna "{VARIABLES_COLUMN}".'
    )

variables_kernel = (
    variables_df[VARIABLES_COLUMN]
    .dropna()
    .astype(str)
    .str.strip()
    .loc[lambda serie: serie.ne("")]
    .tolist()
)
if not variables_kernel:
    raise ValueError(f'La hoja "{VARIABLES_SHEET}" no contiene variables utilizables.')

variables_duplicadas = pd.Index(variables_kernel)[pd.Index(variables_kernel).duplicated()].unique().tolist()
if variables_duplicadas:
    raise ValueError(f"Hay variables duplicadas en {VARIABLES_SHEET}: {variables_duplicadas}")

variables_faltantes = [col for col in variables_kernel if col not in kernel_feature_columns]
if variables_faltantes:
    raise KeyError(f"Variables de la hoja que no existen en muestreos: {variables_faltantes}")

# Se conserva el orden de la hoja y se descartan todas las demas features.
columnas_control = [col for col in kernel_df.columns if col not in kernel_feature_columns]
kernel_feature_columns = variables_kernel
kernel_train_df = kernel_train_df.loc[:, kernel_feature_columns].copy()
kernel_test_df = kernel_test_df.loc[:, kernel_feature_columns].copy()
kernel_df = kernel_df.loc[:, kernel_feature_columns + columnas_control].copy()

# Cada variable se codifica en un qubit; los feature maps y backends toman
# automaticamente esta dimension desde los dataframes filtrados.
N_QUBITS = len(kernel_feature_columns)

print("Feature map:", FEATURE_MAP, "| opciones:", list(FEATURE_MAPS))
print(f"Muestreo train: {TRAIN_MUESTREO} | Muestreo test: {TEST_MUESTREO}")
print(f"Variables seleccionadas ({N_QUBITS}):", kernel_feature_columns)
print(f"Qubits por circuito: {N_QUBITS}")

### Datos que alimentan a los modelos

`cargar_datos_kernel_muestreo` lee la hoja `muestreos` de
`dataset_v1.xlsx`, separa train/test por `_PartInd_` y filtra cada
particion por su nivel de muestreo jerarquico. A continuacion, el cuaderno
lee la hoja `variables` y usa la columna `_vars_` como fuente de verdad:
valida nombres y duplicados, conserva el orden registrado y elimina del
kernel cualquier feature no seleccionada.

`kernel_train_df` construye K_train y `kernel_test_df` construye K_test.
Cada columna seleccionada se codifica en un qubit, por lo que el numero de
qubits se ajusta automaticamente a `len(kernel_feature_columns)`.

Nota: esta hoja no esta alineada fila a fila con
`data/processed/df_escalado.csv` (pipeline del handoff/guppy, escalado
independiente); usar esta funcion implica que el kernel consume las
features del pipeline v1 para las filas seleccionadas.

In [ ]:
# Cantidad de registros y vista previa de las tablas que alimentan el kernel.
print(f"Dataset completo : {kernel_df.shape[0]} registros")
print(f"Train (_PartInd_=0): {kernel_train_df.shape[0]} registros | {kernel_train_df.shape[1]} features")
print(f"Test  (_PartInd_=1): {kernel_test_df.shape[0]} registros | {kernel_test_df.shape[1]} features")
print(f"Dimension cuantica: {N_QUBITS} qubits (uno por feature)")

print("\nTrain (head) -> alimenta K_train:")
display(kernel_train_df.head())
print("Test (head) -> alimenta K_test:")
display(kernel_test_df.head())

## 2. Inspeccion del feature map U(x)

No consume shots. El circuito asigna un qubit por cada variable incluida en
la hoja `variables`.

In [ ]:
PREVIEW_ROW = 0     # Cambia esta fila para inspeccionar otro U(x), sin ejecutar shots

preview_x = kernel_train_df.iloc[PREVIEW_ROW].to_numpy(dtype=float)
feature_map_fn = obtener_feature_map(FEATURE_MAP)
feature_map_preview = feature_map_fn(preview_x)
assert feature_map_preview.n_qubits == N_QUBITS, "El circuito no coincide con las variables seleccionadas."
print(f"Feature map '{FEATURE_MAP}' de train[{PREVIEW_ROW}] | qubits: {feature_map_preview.n_qubits} | puertas: {feature_map_preview.n_gates}")
render_circuit_jupyter(feature_map_preview)

## 3. Seleccion e inspeccion del par

Construye ``U(x_j)^dagger U(x_i)`` con barreras para revisarlo antes de ejecutar.

In [ ]:
KERNEL_ROW_I = 0    # Filas de train que forman el par
KERNEL_ROW_J = 1

kernel_x_i, kernel_x_j, kernel_preview_circuit = seleccionar_par_kernel(
    kernel_train_df, KERNEL_ROW_I, KERNEL_ROW_J, feature_map=FEATURE_MAP
)
render_circuit_jupyter(kernel_preview_circuit)

## 4. Enviar la matriz K_train

`K_train = K(X_train, X_train)` (cuadrada, para `SVC.fit`). Se ejecuta el
triangulo superior y se refleja por simetria; para `m` filas de train se
requieren `m(m-1)/2` circuitos (mas la diagonal si `MATRIX_EXECUTE_DIAGONAL`).

Backends: Selene local o Nexus (Selene, H1/H2 via compile job, Helios). En
local el resultado llega aqui mismo; en Nexus se envia el job y se sigue en
el paso 5. **Tras enviar a Nexus no reejecutes esta celda.**

In [ ]:
# Incluye automaticamente todas las filas disponibles en kernel_train_df.
# Si cambia el nivel de muestreo, MATRIX_ROWS se ajusta sin editar esta celda.
MATRIX_ROWS = list(range(kernel_train_df.shape[0]))
MATRIX_BACKEND = "nexus_selene_statevector"               # Ver MATRIX_BACKEND_OPTIONS
RUN_MATRIX = False                           # Interruptor de seguridad
MATRIX_SHOTS = 1000
MATRIX_SEED = 42
MATRIX_EXECUTE_DIAGONAL = True               # False fija K(i,i)=1 sin ejecutar
SAVE_MATRIX_RUN = True

matrix_state_train, matrix_result_train = iniciar_matriz_kernel(
    kernel_train_df, MATRIX_ROWS, MATRIX_BACKEND, RUN_MATRIX,
    n_shots=MATRIX_SHOTS, seed=MATRIX_SEED,
    ejecutar_diagonal=MATRIX_EXECUTE_DIAGONAL,
    guardar=SAVE_MATRIX_RUN, project_name=PROJECT_NAME,
    feature_map=FEATURE_MAP,
)

if matrix_result_train is not None:
    display(pd.DataFrame(matrix_result_train["kernel_matrix"], index=MATRIX_ROWS, columns=MATRIX_ROWS))
    display(matrix_result_train["run_summary"])

## 5. Consultar y guardar K_train

En Nexus, reejecuta **solo esta celda** hasta que el job llegue a COMPLETED
(H1/H2 encadena compile -> execute automaticamente). En local no hay nada que
consultar (`matrix_state_train` es None) y se guarda directo la matriz del
paso 4. Persiste la matriz de Gram cuadrada como `kernel_qsvm_<run_id>.csv`
(lista para `SVC(kernel="precomputed")`) + metadatos.

In [ ]:
matrix_state_train, matrix_train_remoto = consultar_matriz_nexus(matrix_state_train, guardar=SAVE_MATRIX_RUN)

# Toma la matriz remota si ya llego; si no, la local del paso 4.
if matrix_train_remoto is not None:
    K_train_final = matrix_train_remoto
    fuente_train = f"nexus_{MATRIX_BACKEND}_train"
    id_train = matrix_state_train["job_ref"].id
elif matrix_result_train is not None:
    K_train_final = matrix_result_train
    fuente_train = "local_statevector_train"
    id_train = None
else:
    K_train_final = None
    print("K_train aun no disponible (job remoto en curso o sin construir).")

if K_train_final is not None:
    ruta_train, ruta_train_meta = guardar_kernel_qsvm(K_train_final, source=fuente_train, job_id=id_train)
    print("K_train guardada en:", ruta_train)
    print("Metadatos en:", ruta_train_meta)
    display(pd.read_csv(ruta_train, sep=";", index_col=0))

## 6. Enviar la matriz K_test

`K_test = K(X_test, X_train)` (rectangular n_test x m, para `SVC.predict`).
Reutiliza los mismos parametros del paso 4 (backend, shots, feature map) y las
mismas `MATRIX_ROWS` como columnas. Internamente apila `[test, train]`,
construye la conjunta y recorta el bloque test x train.

Es un **job independiente** del de K_train: en Nexus puedes lanzar este envio
sin esperar a que termine el de train, y consultar cada uno por separado.

In [ ]:
# Incluye automaticamente todas las filas disponibles en kernel_test_df.
# Si cambia el nivel de muestreo, TEST_ROWS se ajusta sin editar esta celda.
TEST_ROWS = list(range(kernel_test_df.shape[0]))

matrix_state_test, matrix_result_test = iniciar_matriz_kernel_test(
    kernel_train_df, MATRIX_ROWS, kernel_test_df, MATRIX_BACKEND, RUN_MATRIX,
    test_rows=TEST_ROWS, n_shots=MATRIX_SHOTS, seed=MATRIX_SEED,
    guardar=SAVE_MATRIX_RUN, project_name=PROJECT_NAME,
    feature_map=FEATURE_MAP,
)

if matrix_result_test is not None:
    display(pd.DataFrame(matrix_result_test["kernel_matrix"], index=TEST_ROWS, columns=MATRIX_ROWS))
    display(matrix_result_test["run_summary"])

## 7. Consultar y guardar K_test

Igual que el paso 5 pero para el job de test. Persiste la matriz rectangular
como `kernel_qsvm_test_<run_id>.csv` (filas = test, columnas = train). Con
K_train (paso 5) y K_test (aqui) ya tienes ambos artefactos para el QSVM:
`SVC(kernel="precomputed").fit(K_train, y_train).predict(K_test)`.

In [ ]:
matrix_state_test, matrix_test_remoto = consultar_matriz_nexus(matrix_state_test, guardar=SAVE_MATRIX_RUN)

if matrix_test_remoto is not None:
    K_test_final = matrix_test_remoto
    fuente_test = f"nexus_{MATRIX_BACKEND}_test"
    id_test = matrix_state_test["job_ref"].id
elif matrix_result_test is not None:
    K_test_final = matrix_result_test
    fuente_test = "local_statevector_test"
    id_test = None
else:
    K_test_final = None
    print("K_test aun no disponible (job remoto en curso o sin construir).")

if K_test_final is not None:
    ruta_test, ruta_test_meta = guardar_kernel_qsvm(K_test_final, source=fuente_test, job_id=id_test)
    print("K_test guardada en:", ruta_test)
    print("Metadatos en:", ruta_test_meta)
    display(pd.read_csv(ruta_test, sep=";", index_col=0))